In [1]:
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sp_moe import SP_MOE

SEED = 5
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
df = pd.read_csv("wildfire.csv")

feature_cols = ["pr", "rmax", "rmin", "srad", "tmmn", "tmmx", "vs", "fm100", "fm1000", "vpd"]
target_col = "erc"

X = df[feature_cols].to_numpy(dtype=np.float32)
y = df[target_col].to_numpy(dtype=np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print(df[feature_cols + [target_col]].describe())

X shape: (502065, 10)
y shape: (502065,)
                  pr           rmax           rmin           srad  \
count  502065.000000  502065.000000  502065.000000  502065.000000   
mean        1.548937      73.304462      30.807555     220.829202   
std         6.157991      21.518443      18.120438      87.354111   
min         0.000000       6.600000       1.000000       0.000000   
25%         0.000000      57.500000      16.900000     147.200000   
50%         0.000000      76.600000      27.300000     223.800000   
75%         0.000000      92.100000      42.100000     298.700000   
max       264.100000     100.000000     100.000000     424.900000   

                tmmn           tmmx             vs          fm100  \
count  502065.000000  502065.000000  502065.000000  502065.000000   
mean      281.517400     295.749225       3.638460      11.875518   
std         7.797477       9.308161       1.664263       4.829020   
min       233.500000     241.900000       0.300000       1.20

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

# fit scalers on train only to avoid leaking test statistics
x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.reshape(-1, 1))

print("train:", X_train_scaled.shape, y_train_scaled.shape)
print("test:", X_test_scaled.shape, y_test_scaled.shape)

train: (401652, 10) (401652, 1)
test: (100413, 10) (100413, 1)


In [4]:
model = SP_MOE(
    input_dim=X_train_scaled.shape[1],
    output_dim=1,
    k=5,
    global_hidden_dims=(64, 64),
    expert_hidden_dims=(64, 64),
    weighting="inverse_distance",
    random_state=SEED,
)

print("device:", model.device)
print("global model:", model.global_model)
print("expert 0:", model.experts[0])

device: cpu
global model: Sequential(
  (0): Linear(in_features=10, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=1, bias=True)
)
expert 0: Sequential(
  (0): Linear(in_features=10, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=1, bias=True)
)


In [9]:
model.fit(
    X_train_scaled,
    y_train_scaled,
    cluster_features=X_train_scaled,
    epochs_global=40,
    epochs_expert=5,
    lr=1e-3,
    batch_size=512,
    verbose=True,
)

[sp_moe] stage 1/3: training global model
[sp_moe] global epoch 1/40 mse=0.002813
[sp_moe] global epoch 8/40 mse=0.002728
[sp_moe] global epoch 16/40 mse=0.002674
[sp_moe] global epoch 24/40 mse=0.002661
[sp_moe] global epoch 32/40 mse=0.002629
[sp_moe] global epoch 40/40 mse=0.002597
[sp_moe] stage 2/3: clustering
[sp_moe] cluster sizes: [10310, 89861, 101323, 105655, 94503]
[sp_moe] stage 3/3: warm-starting and fine-tuning experts
[sp_moe] expert 0: fine-tuning on 10310 samples
[sp_moe] expert 1: fine-tuning on 89861 samples
[sp_moe] expert 2: fine-tuning on 101323 samples
[sp_moe] expert 3: fine-tuning on 105655 samples
[sp_moe] expert 4: fine-tuning on 94503 samples


SP_MOE(
  (global_model): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
  (experts): ModuleList(
    (0-4): 5 x Sequential(
      (0): Linear(in_features=10, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU()
      (4): Linear(in_features=64, out_features=1, bias=True)
    )
  )
)

In [10]:
y_pred_scaled, weights = model.predict(X_test_scaled, cluster_features=X_test_scaled, return_weights=True)
y_pred = y_scaler.inverse_transform(y_pred_scaled).ravel()

y_pred_global_scaled = model.predict_global(X_test_scaled)
y_pred_global = y_scaler.inverse_transform(y_pred_global_scaled).ravel()

def report(name, y_true, y_hat):
    rmse = mean_squared_error(y_true, y_hat) ** 0.5
    mae = mean_absolute_error(y_true, y_hat)
    r2 = r2_score(y_true, y_hat)
    print(f"{name:>12s} | rmse={rmse:.4f}  mae={mae:.4f}  r2={r2:.4f}")

report("sp_moe", y_test, y_pred)
report("global only", y_test, y_pred_global)

print("\nmean gating weight per expert:", weights.mean(axis=0).round(3))
print("test cluster sizes:", np.bincount(model.assign_clusters(X_test_scaled), minlength=model.k))

      sp_moe | rmse=1.2426  mae=0.6257  r2=0.9973
 global only | rmse=1.2083  mae=0.5890  r2=0.9975

mean gating weight per expert: [0.059 0.241 0.213 0.256 0.231]
test cluster sizes: [ 2545 22457 25575 26159 23677]


In [12]:
hard_labels = model.assign_clusters(X_test_scaled)
for i in range(model.k):
    mask = hard_labels == i
    moe_rmse = mean_squared_error(y_test[mask], y_pred[mask]) ** 0.5
    glob_rmse = mean_squared_error(y_test[mask], y_pred_global[mask]) ** 0.5
    print(f"cluster {i} (n={mask.sum():6d}): sp_moe rmse={moe_rmse:.4f}  global rmse={glob_rmse:.4f}")

cluster 0 (n=  2545): sp_moe rmse=1.2229  global rmse=0.7920
cluster 1 (n= 22457): sp_moe rmse=1.3231  global rmse=1.3280
cluster 2 (n= 25575): sp_moe rmse=1.2001  global rmse=1.1521
cluster 3 (n= 26159): sp_moe rmse=1.1376  global rmse=1.1214
cluster 4 (n= 23677): sp_moe rmse=1.3199  global rmse=1.2762
